# Week 5: Autoencoders & Embeddings - Homework

**ML2: Advanced Machine Learning**

**Estimated Time**: 1 hour

---

This homework combines programming exercises and knowledge-based questions to reinforce this week's concepts.

## Setup

Run this cell to import necessary libraries:

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

# Set random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print('✓ Libraries imported successfully')

✓ Libraries imported successfully


---
## Part 1: Programming Exercises (60%)

Complete the following programming tasks. Read each description carefully and implement the requested functionality.

### Exercise 1: Experiment: Compression Forces Learning

**Time**: 12 min

Observe how different bottleneck sizes affect reconstruction quality and feature learning.

In [2]:
import torch
import torch.nn as nn
import torchvision
import matplotlib.pyplot as plt

class Autoencoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 784),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded, encoded

# Train autoencoders with different bottleneck sizes
latent_dims = [2, 8, 32, 128]

transform = transforms.Compose([transforms.ToTensor()])
train_data = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_data, batch_size=256, shuffle=True)

results = {}

for latent_dim in latent_dims:
    model = Autoencoder(latent_dim)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.MSELoss()
    
    for epoch in range(5):
        total_loss = 0
        for imgs, _ in train_loader:
            imgs = imgs.view(imgs.size(0), -1)
            optimizer.zero_grad()
            recon, _ = model(imgs)
            loss = criterion(recon, imgs)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(train_loader)
    
    results[latent_dim] = avg_loss
    print(f"latent_dim={latent_dim}: final loss={avg_loss:.4f}")

latent_dim=2: final loss=0.0470
latent_dim=8: final loss=0.0237
latent_dim=32: final loss=0.0110
latent_dim=128: final loss=0.0087


---
## Part 2: Knowledge Questions (40%)

Answer the following questions to test your conceptual understanding.

### Question 1 (Short Answer)

**Question 1 - Why Compression Forces Learning**

An autoencoder with 784-dim input → 32-dim latent → 784-dim output must compress 784 numbers into 32.

Explain:
1. Why can't the model just memorize each input?
2. What must it learn instead?
3. What happens if latent_dim = 784 (no compression)?

**Hint**: Compression = information bottleneck. The model must learn the ESSENCE of the data.

**Your Answer**:

With only 32 dimensions to store 784 pixels of information, the model physically cannot memorize individual inputs. Instead it must learn shared structure across the dataset, things like strokes, curves, and shapes that are common to all digits. With latent_dim=784 there is no compression pressure, so the model can learn a near-identity mapping without discovering any useful features.

### Question 2 (Short Answer)

**Question 2 - Latent Space as Learned Representation**

After training an autoencoder on MNIST digits, the 32-dimensional latent space captures what makes each digit unique.

Explain:
1. Why might similar digits (like 3 and 8) be close in latent space?
2. How is this different from pixel space (raw 784 dimensions)?
3. What makes the latent representation 'better' than raw pixels?

**Hint**: Latent space captures semantic similarity, not just pixel similarity.

**Your Answer**:

3 and 8 share visual structure (curves, similar stroke patterns), so the encoder maps them to nearby regions of latent space. In pixel space, two 3s written by different people might be far apart even though they're the same digit. The latent space captures semantic identity rather than raw pixel arrangement, making it more useful for downstream tasks like classification or retrieval.

### Question 3 (Multiple Choice)

**Question 3 - Reconstruction Loss**

You train an autoencoder and get: Train reconstruction loss = 0.01, Test reconstruction loss = 0.10

What does this suggest?

A) The model is working perfectly
B) The model is overfitting
C) The latent dimension is too large
D) The model needs more training

A) The model is working perfectly
B) The model is overfitting
C) The latent dimension is too large
D) The model needs more training

**Hint**: Large gap between train and test = overfitting.

**Your Answer**: B

**Explanation**: The 10x gap between train and test loss means the model is memorizing training images rather than learning generalizable structure.

### Question 4 (Short Answer)

**Question 4 - Autoencoders vs Supervised Learning**

Autoencoders are UNSUPERVISED - they don't need labels.

Explain:
1. What is the 'label' that an autoencoder trains on?
2. Why is this useful when you don't have labeled data?
3. How could you use an autoencoder's learned representations for a downstream supervised task?

**Hint**: The input IS the label (reconstruct yourself). The latent space can be used for other tasks.

**Your Answer**:

The label is the input itself: the model tries to reconstruct its own input, so no human annotation is needed. This lets you train on large amounts of unlabeled data to learn useful representations. For a downstream task, you freeze the encoder and attach a small classifier to the latent vectors, training it on whatever labeled data you do have.

### Question 5 (Short Answer)

**Question 5 - VAE vs Standard Autoencoder**

Variational Autoencoders (VAEs) learn a DISTRIBUTION in latent space, not just a point.

Explain:
1. Why is learning a distribution useful for GENERATION?
2. What can VAEs do that standard autoencoders cannot?
3. What's the tradeoff?

**Hint**: Distribution = you can sample new points. Standard AE only encodes existing data.

**Your Answer**:

Learning a distribution means you can sample new latent vectors from it and decode them into plausible new images, not just reconstruct existing ones. A standard autoencoder maps inputs to fixed points and has no principled way to sample novel data. The tradeoff is that the probabilistic constraint (KL divergence regularization) makes reconstructions slightly blurrier.

### Question 6 (Multiple Choice)

**Question 6 - Bottleneck Size Selection**

You're building an autoencoder for 1000x1000 images. Which latent dimension is most reasonable?

A) latent_dim = 2 (extreme compression)
B) latent_dim = 256 (moderate compression)
C) latent_dim = 1000000 (no compression)
D) latent_dim = 100000 (minimal compression)

A) latent_dim = 2 (extreme compression)
B) latent_dim = 256 (moderate compression)
C) latent_dim = 1000000 (no compression)
D) latent_dim = 100000 (minimal compression)

**Hint**: Too small = loss of information. Too large = no compression benefit. Need balance.

**Your Answer**: B

**Explanation**: 256 dimensions provides meaningful compression of a 1M-pixel image while preserving enough capacity to reconstruct important features. 2 dimensions would destroy most information, and 100k-1M provides almost no compression benefit.

### Question 7 (Short Answer)

**Question 7 - Denoising Autoencoders**

A denoising autoencoder is trained with: corrupted_input → encoder → decoder → clean_output

Explain:
1. Why does this make the learned features MORE robust?
2. What additional capability does the model gain?
3. How is this related to data augmentation?

**Hint**: Learning to denoise forces the model to learn the underlying structure, not memorize noise.

**Your Answer**:

To recover a clean image from a corrupted one, the model can't rely on memorizing pixel values and must learn the underlying structure of the data. This gives it the added ability to actually denoise new corrupted inputs at inference time. It's related to data augmentation in that the corruption artificially expands the training distribution, exposing the model to more input variation.

### Question 8 (Short Answer)

**Question 8 - Embeddings as Dimensionality Reduction**

Autoencoder latent space, PCA, and t-SNE all reduce dimensionality. 

Compare:
1. How does an autoencoder differ from PCA?
2. When would you prefer an autoencoder over PCA?
3. What's the computational tradeoff?

**Hint**: PCA = linear. Autoencoder = nonlinear (with activation functions). PCA is faster.

**Your Answer**:

PCA finds the best linear projection; autoencoders can learn nonlinear manifolds because of their activation functions. An autoencoder is better when the data has complex nonlinear structure that a linear projection would fail to capture. The tradeoff is cost: PCA has a closed-form solution while an autoencoder requires iterative training.

### Question 9 (Short Answer)

**Question 9 - Interpolation in Latent Space**

You encode two images to latent vectors z1 and z2. Then you decode MIDPOINT (z1 + z2)/2.

What do you expect to see? Why is this useful?

**Hint**: If latent space is smooth, the midpoint should be a blend of the two images.

**Your Answer**:

You'd expect a blended image that looks like a mix of both inputs, since the latent space is continuous and the decoder maps nearby points to visually similar outputs. This is useful for data augmentation, understanding what the model has learned, and generating smooth transitions between examples.

### Question 10 (Short Answer)

**Question 10 - Real-World Application**

Google Photos uses learned embeddings to search photos by similarity without tags.

Explain:
1. How does an autoencoder-style approach enable this?
2. Why is pixel-space similarity not good enough?
3. What must the latent space capture to make semantic search work?

**Hint**: Latent space must capture 'what the image contains' not 'what pixels look like'.

**Your Answer**:

The encoder maps each photo to a compact embedding and similar-looking photos end up close together, so you can search by nearest neighbor in embedding space. Pixel similarity fails because the same subject in different lighting, angles, or backgrounds will look very different at the pixel level. The latent space needs to capture semantic content like objects, scenes, and people rather than surface-level pixel patterns.

---
## Submission

Before submitting:
1. Run all cells to ensure code executes without errors
2. Check that all questions are answered
3. Review your explanations for clarity

**To Submit**:
- File → Download → Download .ipynb
- Submit the notebook file to your course LMS

**Note**: Make sure your name is in the filename (e.g., homework_01_yourname.ipynb)